# LSTM + FastText (PyTorch) 
**Changes vs original (imbalance handling intentionally left untouched):**
1. Added padding mask / sequence-length handling via `pack_padded_sequence` so the RNN skips padded timesteps.
2. Unfroze the FastText embedding layer (`requires_grad = True`).
3. Removed the `MAX_VOCAB = 5000` cap — full vocabulary is kept.
4. Random search restored (same as your original random-search block).

In [ ]:
import os
import random
import itertools
import numpy as np
import pandas as pd
import warnings
from collections import Counter
from dotenv import load_dotenv
warnings.filterwarnings('ignore')
load_dotenv()

import torch
import torch.nn as nn
from torch.nn.utils.rnn import pack_padded_sequence
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import StratifiedGroupKFold, train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score,
    recall_score, roc_auc_score, classification_report
)
import fasttext
import wandb

# ── Reproducibility ───────────────────────────────────────────────
RANDOM_SEED = 42
os.environ['PYTHONHASHSEED'] = str(RANDOM_SEED)
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
torch.cuda.manual_seed_all(RANDOM_SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print('Libraries loaded.')
print(f'PyTorch version: {torch.__version__}')
print(f'Device: {DEVICE}')

In [ ]:
wandb_key = os.environ.get('WANDB_API_KEY')
if not wandb_key:
    raise ValueError('Set WANDB_API_KEY in your .env file')

wandb.login(key=wandb_key)
wandb.init(
    project = 'commitment-mining',
    name    = 'dl-lstm-fasttext-aug',
    config  = {'model': 'LSTM', 'embedding': 'FastText', 'framework': 'pytorch',
               'fixes': ['masking', 'unfrozen_embeddings', 'uncapped_vocab']},
    tags    = ['lstm', 'deep-learning', 'fasttext', 'pytorch', 'augmented']
)
print('WandB initialized.')

In [ ]:
TRAIN_PATH = 'Commitment-Mining/dataset/train_80p-aug.xlsx'
TEST_PATH  = 'Commitment-Mining/dataset/test_20p.xlsx'
TEXT_COL   = 'statements (ne)'
LABEL_COL  = 'final_label'

train_df = pd.read_excel(TRAIN_PATH)
test_df  = pd.read_excel(TEST_PATH)

for df in [train_df, test_df]:
    df[TEXT_COL]  = df[TEXT_COL].astype(str).str.strip()
    df[LABEL_COL] = df[LABEL_COL].astype(str).str.strip()

print(f'Train size : {len(train_df)}')
print(f'Test size  : {len(test_df)}')
print('\nTrain label distribution:')
print(train_df[LABEL_COL].value_counts())

label_counts = train_df[LABEL_COL].value_counts().to_dict()
wandb.log({'train_size': len(train_df), 'test_size': len(test_df), **label_counts})

In [ ]:
train_df['sentence_length'] = train_df[TEXT_COL].apply(
    lambda x: pd.cut(
        [len(x.split())],
        bins=[0, 5, 10, 20, 50, 999],
        labels=['xs', 's', 'm', 'l', 'xl']
    )[0]
)

train_df['strat_key'] = (
    train_df['province'].astype(str)           + '_' +
    train_df['sentence_length'].astype(str)    + '_' +
    train_df['district/gaupalika'].astype(str) + '_' +
    train_df[LABEL_COL].astype(str)
)

counts = train_df['strat_key'].value_counts()
rare   = counts[counts < 5].index
train_df['strat_key'] = train_df['strat_key'].apply(
    lambda x: 'rare' if x in rare else x
)

print(f'Unique strat keys : {train_df["strat_key"].nunique()}')

In [ ]:
le      = LabelEncoder()
y_train = le.fit_transform(train_df[LABEL_COL].values)
y_test  = le.transform(test_df[LABEL_COL].values)
groups  = train_df['strat_key'].values

X_train_text = [str(x) for x in train_df[TEXT_COL].tolist()]
X_test_text  = [str(x) for x in test_df[TEXT_COL].tolist()]

print(f'Classes  : {le.classes_}')
print(f'y_train  : {y_train.shape} | dtype: {y_train.dtype}')

## FastText embeddings 

**Previously:** the generic pretrained `cc.ne.300.bin` (Common Crawl Nepali) was used as-is — never adapted to this dataset's domain/vocabulary, unlike the TF script which trains FastText directly on the task data.

**Fix:** train FastText **unsupervised on `X_train_text` only** (never on val/test text, to avoid the leakage issue discussed earlier), mirroring the TF script's `fasttext.train_unsupervised(...)` call. This produces domain-adapted embeddings instead of relying solely on generic pretrained vectors.

Combined with the unfrozen embedding layer added earlier, this gives the model both:
1. Domain-adapted starting vectors (from this cell)
2. Task-adapted fine-tuning during classifier training (from `requires_grad = True`)

Set `USE_DOMAIN_FASTTEXT = False` below to instead fall back to the generic pretrained `cc.ne.300.bin`, if you want to compare both.

In [ ]:
import os
import fasttext

EMBED_DIM = 300
USE_DOMAIN_FASTTEXT = False        # domain-adapted (now pretrained-initialized) vs pure generic
FASTTEXT_VEC_PATH = '/home/rupak/Desktop/Commitment-Mining/embeddings/cc.ne.300.vec'
FASTTEXT_BIN_PATH = '/home/rupak/Desktop/Commitment-Mining/embeddings/cc.ne.300.bin'

if USE_DOMAIN_FASTTEXT:
    print('Training FastText on training data, initialized from pretrained cc.ne.300 vectors...')

    # Generate the .vec file from the .bin once, if it doesn't exist yet
    if not os.path.exists(FASTTEXT_VEC_PATH):
        print('  .vec not found — extracting it from .bin (one-time, may take a few minutes)...')
        pretrained_model = fasttext.load_model(FASTTEXT_BIN_PATH)
        words = pretrained_model.get_words()
        dim = pretrained_model.get_dimension()
        with open(FASTTEXT_VEC_PATH, 'w', encoding='utf-8') as f:
            f.write(f"{len(words)} {dim}\n")
            for w in words:
                vec_str = ' '.join(map(str, pretrained_model.get_word_vector(w)))
                f.write(f"{w} {vec_str}\n")
        del pretrained_model  # free memory before the next training step
        print(f'  Saved {FASTTEXT_VEC_PATH}')

    FASTTEXT_TRAIN_CORPUS = 'fasttext_train_corpus.txt'
    with open(FASTTEXT_TRAIN_CORPUS, 'w', encoding='utf-8') as f:
        for text in X_train_text:      # TRAIN TEXT ONLY -- no val/test leakage
            f.write(text + '\n')

    ft_model = fasttext.train_unsupervised(
        FASTTEXT_TRAIN_CORPUS,
        model='skipgram',
        dim=EMBED_DIM,
        epoch=20,
        lr=0.05,
        wordNgrams=2,
        minCount=3,
        ws=5,
        thread=4,
        pretrainedVectors=FASTTEXT_VEC_PATH   # <-- the actual change
    )
    ft_model.save_model('trained_fasttext_model_pytorch.bin')
    print('Pretrained-initialized, domain-fine-tuned FastText model trained and saved.')

    if os.path.exists(FASTTEXT_TRAIN_CORPUS):
        os.remove(FASTTEXT_TRAIN_CORPUS)
else:
    print('Loading generic pretrained FastText model (no domain fine-tuning)...')
    ft_model = fasttext.load_model(FASTTEXT_BIN_PATH)

print(f'FastText ready. Embedding dim: {EMBED_DIM}')

## Tokenizer, padding, and embedding matrix

**FIX #3:** `MAX_VOCAB` cap removed — the tokenizer now keeps the full vocabulary instead of discarding everything past the top 5000 words into `<OOV>`.

**FIX #1 (part 1):** `pad_sequences` now also returns the true (unpadded) length of every sequence — required later to mask the RNN properly.

In [ ]:
MAX_LEN   = 64
OOV_TOKEN = '<OOV>'
PAD_IDX   = 0
OOV_IDX   = 1
# NOTE: MAX_VOCAB cap removed (was 5000) — full vocabulary is kept now.

class SimpleTokenizer:
    """Minimal replacement for tf.keras.preprocessing.text.Tokenizer.
    Index 0 is reserved for padding, index 1 for OOV.
    FIX: num_words is now optional; None keeps the entire vocabulary.
    """
    def __init__(self, num_words=None, oov_token=OOV_TOKEN):
        self.num_words = num_words
        self.oov_token = oov_token
        self.word_index = {}

    def fit_on_texts(self, texts):
        counter = Counter()
        for t in texts:
            counter.update(t.split())
        self.word_index[self.oov_token] = OOV_IDX

        most_common = counter.most_common() if self.num_words is None \
            else counter.most_common(self.num_words - 2)

        for i, (word, _) in enumerate(most_common):
            self.word_index[word] = i + 2  # 0=pad, 1=oov

    def texts_to_sequences(self, texts):
        sequences = []
        for t in texts:
            seq = [self.word_index.get(w, OOV_IDX) for w in t.split()]
            sequences.append(seq)
        return sequences


def pad_sequences(sequences, maxlen, padding='post', truncating='post'):
    """FIX: also returns true sequence lengths, needed for RNN masking."""
    out = np.zeros((len(sequences), maxlen), dtype=np.int64)
    lengths = np.zeros(len(sequences), dtype=np.int64)
    for i, seq in enumerate(sequences):
        if len(seq) > maxlen:
            seq = seq[:maxlen] if truncating == 'post' else seq[-maxlen:]
        lengths[i] = max(len(seq), 1)  # avoid zero-length sequences
        if padding == 'post':
            out[i, :len(seq)] = seq
        else:
            out[i, maxlen - len(seq):] = seq
    return out, lengths


# Fit tokenizer on ALL training text (uncapped vocabulary)
tokenizer = SimpleTokenizer(num_words=None, oov_token=OOV_TOKEN)
tokenizer.fit_on_texts(X_train_text)
vocab_size = len(tokenizer.word_index) + 1  # +1 for pad index 0

# Pad sequences (now also returns lengths)
X_train_seq, len_train = pad_sequences(
    tokenizer.texts_to_sequences(X_train_text), maxlen=MAX_LEN, padding='post', truncating='post'
)
X_test_seq, len_test = pad_sequences(
    tokenizer.texts_to_sequences(X_test_text), maxlen=MAX_LEN, padding='post', truncating='post'
)

print(f'Vocab size (uncapped) : {vocab_size}')
print(f'X_train_seq    : {X_train_seq.shape}')
print(f'X_test_seq     : {X_test_seq.shape}')

# Build embedding matrix
print('Building embedding matrix...')
embedding_matrix = np.zeros((vocab_size, EMBED_DIM), dtype=np.float32)
for word, idx in tokenizer.word_index.items():
    if idx < vocab_size:
        embedding_matrix[idx] = ft_model.get_word_vector(word)

print(f'Embedding matrix : {embedding_matrix.shape}')

wandb.log({
    'vocab_size' : vocab_size,
    'max_len'    : MAX_LEN,
    'embed_dim'  : EMBED_DIM
})

## Random search space (unchanged from original)

In [ ]:
param_distributions = {
    'rnn_units'   : [36, 48, 64],
    'dropout'     : [0.3, 0.4, 0.5],
    'dense_units' : [24, 32],
    'learning_rate': [1e-4, 3e-4, 5e-4],
    'batch_size'  : [16, 32],
    'l2_reg'      : [0.0005, 0.01, 0.001]   
}
N_ITER = 20  

random.seed(RANDOM_SEED)
sampled_configs = [
    {k: random.choice(v) for k, v in param_distributions.items()}
    for _ in range(N_ITER)
]

print(f'Total configs to try : {N_ITER}')
print('Sample config 1:', sampled_configs[0])

## Model, dataset, and training loop

**FIX #1 (part 2):** `SequenceDataset` now also carries the true sequence length. Inside `forward`, embeddings are packed with `pack_padded_sequence` before entering the RNN, so padded timesteps are skipped entirely (equivalent to Keras' `mask_zero=True`).

**FIX #2:** `self.embedding.weight.requires_grad = True` — embeddings are now trainable instead of frozen.

Loss function (`BCEWithLogitsLoss`, no `pos_weight`) and lack of augmentation are left exactly as in the original — imbalance handling untouched.

In [ ]:
class SequenceDataset(Dataset):
    """FIX: now also stores true sequence lengths for masking."""
    def __init__(self, X, lengths, y):
        self.X = torch.as_tensor(X, dtype=torch.long)
        self.lengths = torch.as_tensor(lengths, dtype=torch.long)
        self.y = torch.as_tensor(y, dtype=torch.float32)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.lengths[idx], self.y[idx]


class LSTMClassifier(nn.Module):
    """Equivalent to:
    Embedding(FastText weights, trainable) -> LSTM (masked) -> Dropout
    -> Dense(relu) -> Dropout -> Dense(1, sigmoid)
    """
    def __init__(self, vocab_size, embed_dim, embedding_matrix, rnn_units, dropout):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=PAD_IDX)
        self.embedding.weight.data.copy_(torch.tensor(embedding_matrix, dtype=torch.float32))
        # FIX: embeddings are now trainable (previously frozen)
        self.embedding.weight.requires_grad = False

        self.rnn = nn.LSTM(embed_dim, rnn_units, batch_first=True, bidirectional=False)
        self.rnn_dropout = nn.Dropout(dropout)

        
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(dropout)
        self.fc2 = nn.Linear(rnn_units, 1)

    def forward(self, x, lengths):
        emb = self.embedding(x)  # (B, L, E)

        # FIX: pack the sequence so the RNN skips padding timesteps entirely
        packed = pack_padded_sequence(
            emb, lengths.cpu(), batch_first=True, enforce_sorted=False
        )
        _, (h_n, c_n) = self.rnn(packed)
        h = torch.cat([h_n[0]], dim=1)

        h = self.rnn_dropout(h)
        
        h = self.dropout(h)
        logits = self.fc2(h).squeeze(-1)  # raw logits, sigmoid applied via loss/prediction
        return logits


def build_model(config):
    model = LSTMClassifier(
        vocab_size, EMBED_DIM, embedding_matrix,
        rnn_units   = config['rnn_units'],
        dropout     = config['dropout'],
        # dense_units = config['dense_units']
    ).to(DEVICE)
    return model


def train_model(model, X_tr, len_tr, y_tr, X_val, len_val, y_val, config,
                 epochs=30, patience=3, verbose=0):
    """Mirrors model.fit(..., callbacks=[EarlyStopping(monitor='val_loss',
    patience=5, restore_best_weights=True)]) from Keras.
    Loss/imbalance handling intentionally unchanged."""
    optimizer = torch.optim.Adam(
        (p for p in model.parameters() if p.requires_grad),
        lr=config['learning_rate'],
        weight_decay=config['l2_reg']
    )
    criterion = nn.BCEWithLogitsLoss()  # unchanged — no pos_weight

    train_loader = DataLoader(
        SequenceDataset(X_tr, len_tr, y_tr),
        batch_size=config['batch_size'], shuffle=True
    )
    X_val_t = torch.as_tensor(X_val, dtype=torch.long).to(DEVICE)
    len_val_t = torch.as_tensor(len_val, dtype=torch.long)
    y_val_t = torch.as_tensor(y_val, dtype=torch.float32).to(DEVICE)

    best_val_loss = float('inf')
    best_state = None
    patience_ctr = 0
    train_losses, val_losses = [], []

    for epoch in range(epochs):
        model.train()
        running_loss, n_seen = 0.0, 0
        for xb, lb, yb in train_loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            optimizer.zero_grad()
            logits = model(xb, lb)
            loss = criterion(logits, yb)
            loss.backward()
            optimizer.step()
            running_loss += loss.item() * len(xb)
            n_seen += len(xb)
        train_loss = running_loss / n_seen

        model.eval()
        with torch.no_grad():
            val_logits = model(X_val_t, len_val_t)
            val_loss = criterion(val_logits, y_val_t).item()

        train_losses.append(train_loss)
        val_losses.append(val_loss)

        if verbose:
            print(f'  Epoch {epoch+1:2d}/{epochs} | loss: {train_loss:.4f} | val_loss: {val_loss:.4f}')

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
            patience_ctr = 0
        else:
            patience_ctr += 1
            if patience_ctr >= patience:
                break

    if best_state is not None:
        model.load_state_dict(best_state)  # restore_best_weights=True

    return train_losses, val_losses


def predict_proba(model, X, lengths, batch_size=256):
    model.eval()
    probs = []
    X_t = torch.as_tensor(X, dtype=torch.long)
    len_t = torch.as_tensor(lengths, dtype=torch.long)
    with torch.no_grad():
        for i in range(0, len(X_t), batch_size):
            xb = X_t[i:i+batch_size].to(DEVICE)
            lb = len_t[i:i+batch_size]
            logits = model(xb, lb)
            probs.append(torch.sigmoid(logits).cpu().numpy())
    return np.concatenate(probs)

## Random search over hyperparameters 

Uses avg folds of `StratifiedGroupKFold` to evaluate each sampled config, same as your original script - now using the masked model and length-aware train/predict functions.

In [ ]:
# Same sgkf as ML models — RANDOM_SEED=42 guarantees identical splits
sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)

# Get fold 1 indices for random search
splits = list(sgkf.split(X_train_seq, y_train, groups=groups))
tr_idx_rs, val_idx_rs = splits[0]   # use fold 1 for hyperparameter search

X_tr_rs   = X_train_seq[tr_idx_rs]
len_tr_rs = len_train[tr_idx_rs]
y_tr_rs   = y_train[tr_idx_rs]

X_val_rs   = X_train_seq[val_idx_rs]
len_val_rs = len_train[val_idx_rs]
y_val_rs   = y_train[val_idx_rs]

print(f'Random search train : {X_tr_rs.shape}')
print(f'Random search val   : {X_val_rs.shape}')
print(f'\nSearching {N_ITER} configs...')

search_results = []

for i, config in enumerate(sampled_configs):
    fold_f1s = []
    for tr_idx, val_idx in splits:   # all 5 folds, not just splits[0]
        X_tr, X_val = X_train_seq[tr_idx], X_train_seq[val_idx]
        len_tr, len_val = len_train[tr_idx], len_train[val_idx]
        y_tr, y_val = y_train[tr_idx], y_train[val_idx]

        torch.manual_seed(RANDOM_SEED)
        model = build_model(config)
        train_model(model, X_tr, len_tr, y_tr, X_val, len_val, y_val, config,
                    epochs=30, patience=3, verbose=0)

        y_proba_val = predict_proba(model, X_val, len_val)
        y_pred_val = (y_proba_val > 0.5).astype(int)
        fold_f1s.append(f1_score(y_val, y_pred_val, average='macro', zero_division=0))

    mean_val_f1 = np.mean(fold_f1s)
    search_results.append({'config': config, 'val_f1': mean_val_f1, 'fold_f1s': fold_f1s})
    print(f'Config {i+1:2d}/{N_ITER} | val_f1: {mean_val_f1:.4f} | {config}')

    wandb.log({f'search/config_{i+1}_val_f1': mean_val_f1, **{f'search/config_{i+1}_{k}': v for k, v in config.items()}})

best_result = max(search_results, key=lambda x: x['val_f1'])
best_config = best_result['config']

print(f'\n✓ Random search complete.')
print(f'Best val F1  : {best_result["val_f1"]:.4f}')
print(f'Best config  : {best_config}')

wandb.config.update({
    'best_search_val_f1': best_result['val_f1'],
    **{f'best_param/{k}': v for k, v in best_config.items()}
})

## 5-fold cross-validation with best config

In [ ]:
def compute_metrics(y_true, y_pred, y_proba):
    return {
        'accuracy'  : accuracy_score(y_true, y_pred),
        'precision' : precision_score(y_true, y_pred, average='macro', zero_division=0),
        'recall'    : recall_score(y_true, y_pred, average='macro', zero_division=0),
        'f1'        : f1_score(y_true, y_pred, average='macro', zero_division=0),
        'auroc'     : roc_auc_score(y_true, y_proba)
    }

fold_metrics = []
all_histories = []

for fold, (tr_idx, val_idx) in enumerate(splits):
    X_tr,   X_val   = X_train_seq[tr_idx],  X_train_seq[val_idx]
    len_tr, len_val = len_train[tr_idx],    len_train[val_idx]
    y_tr,   y_val   = y_train[tr_idx],      y_train[val_idx]

    torch.manual_seed(RANDOM_SEED)
    model = build_model(best_config)

    train_loss_hist, val_loss_hist = train_model(
        model, X_tr, len_tr, y_tr, X_val, len_val, y_val, best_config,
        epochs=30, patience=3, verbose=0
    )

    all_histories.append((train_loss_hist, val_loss_hist))

    y_proba = predict_proba(model, X_val, len_val)
    y_pred  = (y_proba > 0.5).astype(int)

    m = compute_metrics(y_val, y_pred, y_proba)
    m['fold'] = fold + 1
    fold_metrics.append(m)

    print(f"Fold {fold+1} | "
          f"Acc: {m['accuracy']:.4f} | "
          f"Prec: {m['precision']:.4f} | "
          f"Rec: {m['recall']:.4f} | "
          f"F1: {m['f1']:.4f} | "
          f"AUROC: {m['auroc']:.4f}")

for fold, (train_loss, val_loss) in enumerate(all_histories):
    for epoch, (tl, vl) in enumerate(zip(train_loss, val_loss)):
        wandb.log({
            f'fold_{fold+1}/train_loss': tl,
            f'fold_{fold+1}/val_loss'  : vl,
            f'fold_{fold+1}/epoch'     : epoch
        })

fold_df    = pd.DataFrame(fold_metrics).set_index('fold')
mean_row   = fold_df.mean().rename('mean')
std_row    = fold_df.std().rename('std')
cv_summary = pd.concat([fold_df, mean_row.to_frame().T, std_row.to_frame().T])

print('\n=== CV Results (best config) ===')
print(cv_summary.round(4))

wandb.log({
    'cv/accuracy_mean'  : fold_df['accuracy'].mean(),
    'cv/accuracy_std'   : fold_df['accuracy'].std(),
    'cv/precision_mean' : fold_df['precision'].mean(),
    'cv/precision_std'  : fold_df['precision'].std(),
    'cv/recall_mean'    : fold_df['recall'].mean(),
    'cv/recall_std'     : fold_df['recall'].std(),
    'cv/f1_mean'        : fold_df['f1'].mean(),
    'cv/f1_std'         : fold_df['f1'].std(),
    'cv/auroc_mean'     : fold_df['auroc'].mean(),
    'cv/auroc_std'      : fold_df['auroc'].std(),
    'cv_results'        : wandb.Table(dataframe=cv_summary.round(4))
})

## Final holdout test evaluation

In [ ]:
# Carve small val set from full train for early stopping
X_tr_full, X_val_full, len_tr_full, len_val_full, y_tr_full, y_val_full = train_test_split(
    X_train_seq, len_train, y_train,
    test_size    = 0.1,
    stratify     = y_train,
    random_state = RANDOM_SEED
)

torch.manual_seed(RANDOM_SEED)
final_model = build_model(best_config)

train_model(
    final_model, X_tr_full, len_tr_full, y_tr_full, X_val_full, len_val_full, y_val_full, best_config,
    epochs=30, patience=3, verbose=1
)

y_proba_test = predict_proba(final_model, X_test_seq, len_test)
y_pred_test  = (y_proba_test > 0.5).astype(int)

test_metrics = compute_metrics(y_test, y_pred_test, y_proba_test)
np.save('y_pred-ft-lstm-aug-pretrained.npy', y_pred_test)
np.save('y_true-ft-lstm.npy', y_test)

print('=== HOLDOUT TEST SET RESULTS ===')
print(f"  Accuracy  : {test_metrics['accuracy']:.4f}")
print(f"  Precision : {test_metrics['precision']:.4f}")
print(f"  Recall    : {test_metrics['recall']:.4f}")
print(f"  F1        : {test_metrics['f1']:.4f}")
print(f"  AUROC     : {test_metrics['auroc']:.4f}")
print('\nClassification Report:')
print(classification_report(y_test, y_pred_test, target_names=le.classes_))

wandb.log({
    'test/accuracy'  : test_metrics['accuracy'],
    'test/precision' : test_metrics['precision'],
    'test/recall'    : test_metrics['recall'],
    'test/f1'        : test_metrics['f1'],
    'test/auroc'     : test_metrics['auroc'],
})

report_df = pd.DataFrame(
    classification_report(y_test, y_pred_test,
                          target_names=le.classes_, output_dict=True)
).transpose().round(4)
wandb.log({'classification_report': wandb.Table(dataframe=report_df)})

In [ ]:
metrics_order = ['accuracy', 'precision', 'recall', 'f1', 'auroc']

summary = pd.DataFrame({
    'CV Mean' : fold_df[metrics_order].mean().round(4),
    'CV Std'  : fold_df[metrics_order].std().round(4),
    'Test'    : pd.Series(test_metrics)[metrics_order].round(4)
})

print('=== PAPER TABLE — LSTM (FastText, PyTorch, FIXED) ===')
print(summary)
print('\nBest hyperparameters:')
for k, v in best_config.items():
    print(f'  {k}: {v}')

wandb.log({'paper_table': wandb.Table(dataframe=summary)})
wandb.finish()